# Gap imputation — the counterfactual reframe (bidirectional, subspace)

The demand simulation recast as **constrained subspace imputation** (supervisor's
reframe). NOT step-ahead forecasting: we know the data on both sides of the
11:00–14:00 window, and inside it we know the *totals* (net_demand, demand, price,
calendar) at every step — only the 6-source **breakdown** is hidden.

**Why the 2:05 seam disappears:** the old approach ran one-directional through the
window and drifted, then snapped back to the real 14:00 value (that snap = the ramp
violation). Here a **bidirectional** model sees the 14:00 boundary, and a **two-sided
ramp tube** pins the fill to it — every consecutive pair incl. both seams is
ramp-feasible by construction.

**The subspace (verified):** `net_demand = SIGN·sources` to 0.000 MW, so inside the
gap the model is handed the exact *sum* of the six hidden numbers and imputes the
5-dim *split* — constrained by both boundaries + ramp + box + balance.

Pipeline & literature: `imputation/README.md`, `constraints/lit_review.md` themes 8–9.


In [ ]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""   # PRIVATE repo: fine-grained READ-ONLY token (Contents: read)
BRANCH = "claude/model-bottlenecks-constraints-gb1aoj"
import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("energy_modelling"): !git clone -q --branch $BRANCH $url
%cd energy_modelling
!git pull -q
!nvidia-smi -L


## 1 · The bar to beat — interpolation baseline

Mask the real 11–14 window, fill by linear interpolation between boundaries, score
reconstruction WAPE vs measured truth (answer key exists — we hid real data).


In [ ]:
!python imputation/baseline.py
from IPython.display import Markdown, display
display(Markdown(open("imputation/results/baselines.md").read()))


## 2 · Sanity — the two-sided ramp tube is feasible incl. both seams


In [ ]:
!python imputation/constraints.py
!python imputation/gap_data.py


## 3 · Train the bi-LSTM imputer (5 years of masked + perturbed windows)

Random-position masked gaps over the 394k-row train flat. `--perturb` scales demand
inside a share of gaps (the professor's synthetic-data idea) so the model can't
score by boundary-blending and must READ net_demand — the counterfactual-robustness
step. Eval = projected reconstruction WAPE on the 186 real test 11–14 days.


In [ ]:
!python imputation/train.py --epochs 25 --n-train 60000 --perturb 0.5 --hidden 128
display(Markdown("**Trained reconstruction (projected, 186 real test days):**"))
import json
print(json.dumps(json.load(open("imputation/results/bilstm_recon.json")), indent=2))


## 4 · The counterfactual — placebo + +10% demand response

placebo (g=0): measured response must be ≈0. scenario (+10%): feed the raised demand
into the gap's known subspace, re-impute, project — does the fleet deliver the extra
load (capture≈1), and stay feasible incl. both seams?


In [ ]:
!python imputation/scenario_eval.py --g 10


## 5 · What to conclude

- **Reconstruction WAPE < 0.345** (the interp baseline) while constraint-clean ⇒ the
  learned imputer earns its keep — mainly by beating interpolation on hydro/ocgt/steam
  and routing the balance correctly (batteries stay the hard channel).
- **Placebo capture ≈ 0, scenario capture ≈ 1, 0 ramp violations incl. both seams** ⇒
  responds to demand, feasibly, with the 2:05 problem gone by construction.
- Next: GRIN (graph across channels) engine; landing-strip right-boundary for the
  energy-non-neutral case; uncertainty intervals (diffusion). See README.


## 6 · Deliverable — 4-day stacked dispatch graph (imputed gaps, +10%)

Mirrors `demand_simulation/study_stack_4day.py`: 4 consecutive test days, each
11:00–14:00 gap imputed. Three panels — actual / base fill / +10% fill. The stack
is **continuous across every gold edge** (no 2:05 seam), and in the bottom panel it
rises to the raised demand line inside each gap.

In [ ]:
!python imputation/stack_gap.py --days 4 --g 10
from IPython.display import Image, display
display(Image("imputation/results/figure/stack_gap_4day_g10.png"))